# Composition and sequence models for GCN4 speed and strength

This notebook compares a small set of regression models for predicting **speed** and **strength** from the GCN4 activation domain sequence. The requested baselines are included explicitly:

- **linear regression on amino-acid composition**
- **linear regression on one-hot encoded sequence identity**

To see whether less interpretable models help, the notebook also benchmarks a few additional regressors on the same train/test split and 5-fold cross-validation.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter
from scipy.stats import pearsonr
from IPython.display import display

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import ElasticNetCV, LinearRegression, RidgeCV
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sns.set_context('talk')
sns.set_style('white')
RANDOM_STATE = 42
AMINO_ACIDS = list('ACDEFGHIKLMNPQRSTVWY')


## Load the fitted speed and strength values


In [ ]:
exponential_fit = pd.read_csv('../../../output/GCN4_pipeline/speed/exponential_fit.csv', index_col=0)
exponential_fit = exponential_fit[['ADseq', 'A+C', 'k']].rename(
    columns={'A+C': 'strength', 'k': 'speed'}
)

exponential_fit.head()


In [ ]:
summary_df = exponential_fit[['speed', 'strength']].agg(['mean', 'std', 'min', 'max']).T
summary_df['n_sequences'] = len(exponential_fit)
display(summary_df)

fig, axs = plt.subplots(1, 2, figsize=(10, 4), dpi=300)
for ax, column in zip(axs, ['speed', 'strength']):
    sns.histplot(exponential_fit[column], bins=60, ax=ax)
    ax.set_title(column.capitalize())
    ax.set_xlabel(column.capitalize())
    ax.set_ylabel('Count')

sns.despine()
plt.tight_layout()


## Shared feature engineering and evaluation helpers


In [ ]:
def aa_composition(seq):
    counts = Counter(seq)
    length = len(seq)
    return {aa: counts.get(aa, 0) / length for aa in AMINO_ACIDS}


def build_one_hot_features(seqs):
    seq_lengths = pd.Series(seqs).str.len()
    assert seq_lengths.nunique() == 1, 'Sequences must all have the same length'
    seq_length = int(seq_lengths.iloc[0])

    aa_to_idx = {aa: idx for idx, aa in enumerate(AMINO_ACIDS)}
    feature_names = [
        f'pos{pos + 1:02d}_{aa}'
        for pos in range(seq_length)
        for aa in AMINO_ACIDS
    ]

    encoded = np.zeros((len(seqs), seq_length * len(AMINO_ACIDS)), dtype=np.float32)
    for row_idx, seq in enumerate(seqs):
        for pos, aa in enumerate(seq):
            if aa not in aa_to_idx:
                raise ValueError(f'Unexpected amino acid {aa!r} in sequence {seq!r}')
            encoded[row_idx, pos * len(AMINO_ACIDS) + aa_to_idx[aa]] = 1.0

    return pd.DataFrame(encoded, columns=feature_names), seq_length


def safe_pearson(y_true, y_pred):
    if np.std(y_true) == 0 or np.std(y_pred) == 0:
        return np.nan
    return pearsonr(y_true, y_pred)[0]


def metric_row(y_true, y_pred, feature_set, target_name, model_name, split_name):
    return {
        'feature_set': feature_set,
        'target': target_name,
        'model': model_name,
        'split': split_name,
        'r2': r2_score(y_true, y_pred),
        'rmse': mean_squared_error(y_true, y_pred) ** 0.5,
        'pearson_r': safe_pearson(y_true, y_pred),
    }


def evaluate_model(builder, X, y, feature_set, target_name, train_idx, test_idx, cv):
    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]
    y_train = y[train_idx]
    y_test = y[test_idx]

    estimator = builder()
    estimator.fit(X_train, y_train)
    y_test_pred = estimator.predict(X_test)

    cv_estimator = builder()
    y_cv_pred = cross_val_predict(cv_estimator, X, y, cv=cv)

    summary_rows = [
        metric_row(y_test, y_test_pred, feature_set, target_name, estimator.__class__.__name__, 'holdout'),
        metric_row(y, y_cv_pred, feature_set, target_name, estimator.__class__.__name__, '5-fold CV'),
    ]

    return summary_rows, {
        'estimator': estimator,
        'y_test': y_test,
        'y_test_pred': y_test_pred,
        'y_cv_pred': y_cv_pred,
    }


def benchmark_models(X, targets, model_builders, feature_set, train_idx, test_idx, cv):
    rows = []
    artifacts = {}
    for target_name, y in targets.items():
        for model_name, builder in model_builders.items():
            summary_rows, artifact = evaluate_model(
                builder, X, y, feature_set, target_name, train_idx, test_idx, cv
            )
            for row in summary_rows:
                row['model'] = model_name
            rows.extend(summary_rows)
            artifacts[(feature_set, target_name, model_name)] = artifact
    return pd.DataFrame(rows), artifacts


def coefficient_table(model, feature_names, n=12):
    coef = np.asarray(model.coef_).reshape(-1)
    coef_df = pd.DataFrame({'feature': feature_names, 'coef': coef})
    coef_df['abs_coef'] = coef_df['coef'].abs()
    return coef_df.sort_values('abs_coef', ascending=False).head(n)


def plot_prediction_grid(artifacts, feature_set, model_name, title_prefix):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4), dpi=300, sharex=False, sharey=False)
    for ax, target_name in zip(axes, ['speed', 'strength']):
        artifact = artifacts[(feature_set, target_name, model_name)]
        y_true = artifact['y_test']
        y_pred = artifact['y_test_pred']
        lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
        ax.scatter(y_true, y_pred, alpha=0.6, edgecolors='none')
        ax.plot(lims, lims, 'r--', lw=1)
        ax.set_title(
            f"{target_name.capitalize()}\nR²={r2_score(y_true, y_pred):.2f}, r={safe_pearson(y_true, y_pred):.2f}"
        )
        ax.set_xlabel('Observed')
        ax.set_ylabel('Predicted')
    fig.suptitle(f'{title_prefix}: {model_name}', y=1.02)
    sns.despine()
    plt.tight_layout()


## Build the composition and one-hot feature sets


In [ ]:
composition_df = pd.DataFrame(exponential_fit['ADseq'].apply(aa_composition).tolist())
onehot_df, seq_length = build_one_hot_features(exponential_fit['ADseq'])

targets = {
    'speed': exponential_fit['speed'].to_numpy(),
    'strength': exponential_fit['strength'].to_numpy(),
}

train_idx, test_idx = train_test_split(
    np.arange(len(exponential_fit)), test_size=0.2, random_state=RANDOM_STATE
)
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

pd.DataFrame({
    'feature_set': ['composition', 'one_hot_sequence'],
    'n_samples': [len(composition_df), len(onehot_df)],
    'n_features': [composition_df.shape[1], onehot_df.shape[1]],
    'sequence_length': [seq_length, seq_length],
})


## Requested linear-regression baselines

These are the two models you asked for directly: ordinary least-squares regression on **composition** and on **one-hot encoded sequence identity**.


In [ ]:
linear_models = {'Linear regression': lambda: LinearRegression()}

composition_linear_summary, composition_linear_artifacts = benchmark_models(
    composition_df, targets, linear_models, 'composition', train_idx, test_idx, cv
)
onehot_linear_summary, onehot_linear_artifacts = benchmark_models(
    onehot_df, targets, linear_models, 'one_hot_sequence', train_idx, test_idx, cv
)

linear_summary = (
    pd.concat([composition_linear_summary, onehot_linear_summary], ignore_index=True)
      .sort_values(['split', 'target', 'r2'], ascending=[True, True, False])
      .reset_index(drop=True)
)

display(linear_summary)


In [ ]:
plot_prediction_grid(composition_linear_artifacts, 'composition', 'Linear regression', 'Composition baseline')
plot_prediction_grid(onehot_linear_artifacts, 'one_hot_sequence', 'Linear regression', 'One-hot baseline')

print('Top composition coefficients for speed')
display(coefficient_table(
    composition_linear_artifacts[('composition', 'speed', 'Linear regression')]['estimator'],
    composition_df.columns,
))

print('Top composition coefficients for strength')
display(coefficient_table(
    composition_linear_artifacts[('composition', 'strength', 'Linear regression')]['estimator'],
    composition_df.columns,
))

print('Top one-hot coefficients for speed')
display(coefficient_table(
    onehot_linear_artifacts[('one_hot_sequence', 'speed', 'Linear regression')]['estimator'],
    onehot_df.columns,
    n=15,
))

print('Top one-hot coefficients for strength')
display(coefficient_table(
    onehot_linear_artifacts[('one_hot_sequence', 'strength', 'Linear regression')]['estimator'],
    onehot_df.columns,
    n=15,
))


## Additional models

To deprioritize interpretability a bit, this section benchmarks a few stronger or more regularized alternatives on the same feature sets:

- **Ridge regression**
- **Elastic net**
- **Random forest**
- **Gradient boosting**


In [ ]:
alphas = np.logspace(-4, 4, 40)

model_builders = {
    'Linear regression': lambda: LinearRegression(),
    'Ridge': lambda: Pipeline([
        ('scaler', StandardScaler()),
        ('model', RidgeCV(alphas=alphas)),
    ]),
    'Elastic net': lambda: Pipeline([
        ('scaler', StandardScaler()),
        ('model', ElasticNetCV(
            l1_ratio=[0.1, 0.5, 0.9, 1.0],
            alphas=np.logspace(-4, 1, 30),
            cv=5,
            max_iter=20000,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )),
    ]),
    'Random forest': lambda: RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=3,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    'Gradient boosting': lambda: GradientBoostingRegressor(random_state=RANDOM_STATE),
}

composition_summary, composition_artifacts = benchmark_models(
    composition_df, targets, model_builders, 'composition', train_idx, test_idx, cv
)
onehot_summary, onehot_artifacts = benchmark_models(
    onehot_df, targets, model_builders, 'one_hot_sequence', train_idx, test_idx, cv
)

model_summary = (
    pd.concat([composition_summary, onehot_summary], ignore_index=True)
      .sort_values(['split', 'target', 'feature_set', 'r2'], ascending=[True, True, True, False])
      .reset_index(drop=True)
)

display(model_summary)

comparison_pivot = model_summary.pivot_table(
    index=['feature_set', 'target', 'model'],
    columns='split',
    values=['r2', 'rmse', 'pearson_r'],
)
comparison_pivot


In [ ]:
cv_summary = model_summary.query("split == '5-fold CV'").copy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4), dpi=300, sharey=True)
for ax, feature_set in zip(axes, ['composition', 'one_hot_sequence']):
    plot_df = cv_summary.query('feature_set == @feature_set').copy()
    sns.barplot(
        data=plot_df,
        x='model',
        y='r2',
        hue='target',
        ax=ax,
    )
    ax.set_title(feature_set.replace('_', ' ').title())
    ax.set_xlabel('')
    ax.set_ylabel('5-fold CV R²')
    ax.tick_params(axis='x', rotation=30)

sns.despine()
plt.tight_layout()


In [ ]:
best_models = (
    model_summary.query("split == '5-fold CV'")
    .sort_values('r2', ascending=False)
    .groupby(['feature_set', 'target'], as_index=False)
    .first()[['feature_set', 'target', 'model', 'r2', 'rmse', 'pearson_r']]
)

display(best_models)

all_artifacts = {}
all_artifacts.update(composition_artifacts)
all_artifacts.update(onehot_artifacts)

for row in best_models.itertuples(index=False):
    plot_prediction_grid(all_artifacts, row.feature_set, row.model, f"Best {row.feature_set.replace('_', ' ')} model")


## Quick readout

The tables above let you compare whether **composition-only** features are already sufficient, or whether the **position-specific one-hot encoding** adds predictive power for either trait. The `best_models` table is the quickest place to check which combination worked best for **speed** and **strength**.
